In [1]:
# ADK Imports
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.tools import AgentTool, google_search
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.genai import types
from dotenv import load_dotenv
import os
from pathlib import Path


In [2]:
# -------------------------------------------------------------------------
# PATH CONFIGURATION
# -------------------------------------------------------------------------
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent  # experiment_notebooks -> project_root

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

pred_models_dir = project_root / "pred_models_training"
data_dir = project_root / "data"
gen_data_dir = data_dir / "gen_data"
pred_data_dir = data_dir / "pred_data"

# Agents

In [3]:
from pydantic import BaseModel, Field, ConfigDict
from typing import List

# Ensure agents use the same langauge

class FactorAnalysis(BaseModel):
    
    verdict: str = Field(description="The final label (e.g., 'sensational', 'support')")
    confidence: int = Field(description="Confidence score 0-100")
    fcot_reasoning: str = Field(description="2-3 sentence FCoT reasoning.")

class FactCheckFinalReport(BaseModel):
    # 1. High-Level Summary
    final_verdict: str = Field(..., description="The definitive verdict (e.g., Verified Accurate, Misleading, Misinformation, Disinformation, etc.).")
    overall_confidence: int = Field(..., ge=0, le=100, description="Confidence score from 0-100.")
    
    # 2. Human-Centric Explanation (The 'Why')
    verdict_justification: str = Field(
        ..., 
        description="A 1-3 sentence explanation synthesizing why this verdict was reached based on the factor analysis."
    )

    # 3. Agent Metadata
    agents_involved: List[str] = Field(
        default=["Sensationalism_Analyst", "Stance_Analyst", "Context_Veracity_Analyst", 
                 "News_Coverage_Analyst", "Intent_Analyst", "Title_Body_Analyst"],
        description="List of specialized agents that contributed factor data."
    )

    # 4. Detailed Factor Signals
    # These contain the individual verdicts and FCoT reasoning for the audit trail
    sensationalism_signal: FactorAnalysis
    stance_signal: FactorAnalysis
    context_veracity_signal: FactorAnalysis
    news_coverage_signal: FactorAnalysis
    intent_signal: FactorAnalysis
    title_body_signal: FactorAnalysis
# -------------------------------------------------------------------------
# Agent Configurations
# -------------------------------------------------------------------------

# Load the .env file
load_dotenv() 

# Retrieve the key
api_key = os.getenv("GOOGLE_API_KEY")

# Check if it loaded correctly
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found! Make sure you created the .env file.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

## FCOT

In [4]:
sensationalism_agent = Agent(
    name="Sensationalism_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    instruction="""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Precision in identifying emotional manipulation and clickbait architecture.
- **MINIMIZE**: 'Stylistic False Positives' where urgency or technical reporting is misclassified as sensationalism.

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Linguistic Scan**: Isolate high-intensity adjectives, superlatives, and emotional triggers (e.g., "Shocking," "Outrageous," "Final Warning").
2.  **Structural Integrity Check**: Does the Body of the text actually contain the "shocking" information promised by the Headline? Identify any "curiosity gaps."

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - RHETORICAL DECONSTRUCTION
1.  **Inverse Perspective**: Re-read the article while stripping away all adjectives. If the core facts remain significant, it is 'Neutral'. If the article collapses into triviality without the "fluff," it is 'Sensational'.
2.  **Rhetorical Intent**: Determine if the author is informing the reader or attempting to trigger a specific physiological stress response.

### PHASE 3: REFLECTIVE UPDATE (RUM) & RETROSPECTIVE ADJUSTMENT
1.  **Consistency Check**: Compare your findings from Phase 1 and Phase 2. If the language is urgent but the facts are heavy, adjust the label toward 'Neutral'. 
2.  **Final Synthesis**: Populate the `fcot_reasoning` field by explaining the logic behind your final classification.

## CONFIDENCE RUBRIC
- 90-100%: Explicit emotional triggers and clear headline-body baiting.
- 75-89%: Pattern of hyperbolic language is consistent throughout.
- 50-74%: Urgency exists, but may be justified by the gravity of the news topic.
- <50%: Signals are too mixed to provide a reliable classification.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- The verdict must be strictly: [sensational, neutral].
"""
)

In [5]:
stance_agent = Agent(
    name="Stance_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    output_schema=FactorAnalysis,
    instruction= """
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Detection of nuanced rhetorical alignment, bias, or skepticism.
- **MINIMIZE**: Misclassification of "objective reporting" as "denial" or "unbiased" as "support."

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Linguistic Tone Mapping**: Identify the author's voice. Look for "loaded" verbs (e.g., 'claimed' vs 'demonstrated') and specific actor framing.
2.  **Preliminary Stance**: Form an internal hypothesis: Does the narrative arc support the primary claim, attempt to debunk it (Deny), or provide balanced reporting (Neutral)?

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - RHETORICAL DECONSTRUCTION
1.  **Actor Alignment**: Identify which stakeholders are mentioned more or cited with the most authority. 
2.  **Omission Check**: Widen the aperture to look for what is *not* said. Does the text ignore standard counter-arguments?
3.  **Inverse Stance Test**: If you re-read the article from the perspective of an opponent, does it feel like an attack, or an fair representation?

### PHASE 3: REFLECTIVE UPDATE (RUM) & SYNTHESIS
1.  **Reflective Alignment**: Compare your findings from Phase 1 and 2. If the tone is neutral but the actor alignment is biased, adjust the verdict toward 'Support'.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field by explaining the specific rhetorical markers that led to your final classification.

## CONFIDENCE RUBRIC
- 90-100%: Explicit, unambiguous alignment of tone and actor framing.
- 70-89%: Clear trend; minor stylistic nuance doesn't obscure the primary stance.
- 50-69%: Content is balanced or uses "both-sidesism" to obscure a subtle bias.
- <50%: Contradictory signals or text lacks sufficient markers to assign a stance.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- Ensure the verdict strictly matches: [support, deny, neutral].
"""
)

In [6]:
context_agent = Agent(
    name="Context_Veracity_Analyst",
    model=Gemini(
        model="gemini-3-flash-preview",
        thinking_level="HIGH"
    ),
    output_schema=FactorAnalysis,
    description="Tool-free FCoT specialist for internal factual consistency and plausibility.",
    instruction=f"""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Detection of internal logical fallacies, chronological impossibilities, and source-validity gaps.
- **MINIMIZE**: False positives on breaking 2026 news that may seem "implausible" but is actually current events.

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Date/Entity Extraction**: Identify all names, dates, and specific claims.
2.  **Structural Audit**: Check for claims that rely on sources that are not named or are vaguely defined (e.g., "experts say").

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO)
1.  **Plausibility Stress-Test**: Evaluate the claims against the known laws of sciences, economics, and historical precedent. 
2.  **Entity Consistency**: Does a named organization actually have the jurisdiction or capability to do what the article claims? (e.g., The WHO declaring a law in a specific US city).
3.  **Temporal Conflict Check**: Do the dates provided (especially 2025-2026) align with the sequence of events described?

### PHASE 3: REFLECTIVE UPDATE (RUM) & SYNTHESIS
1.  **Reflective Alignment**: If the story is internally consistent but relies on extreme "Shock Value" without specific evidence, adjust confidence downward.
2.  **Final Synthesis**: In `fcot_reasoning`, explain the logical "fail points" that determined the veracity.

## CONFIDENCE RUBRIC
- 90-100%: Claims are internally consistent, logical, and follow a verifiable narrative structure.
- 70-89%: Generally plausible; minor vague areas but no logical "deal-breakers."
- 50-69%: "Red Flag" territory; relies on high-emotion claims with low logical backing.
- <50%: Contains a clear logical or chronological impossibility (e.g., an event happening before its cause).

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- The verdict must be strictly: [Accurate, Inaccurate].
"""
)

In [7]:

news_coverage_agent = Agent(
    name="News_Coverage_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT agent specializing in multi-scale news categorization.",
    instruction="""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Precision in identifying the primary thematic domain and geographical scope.
- **MINIMIZE**: Conceptual redundancy (e.g., mislabeling a 'Political' story as 'General' because it mentions a city name).

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Topical Extraction**: Identify the "Who, What, Where" within the text. 
2.  **Granularity Check**: Determine the scale of the news. Is it a local incident, a national policy, or an international trend?
3.  **Preliminary Classification**: Form an internal hypothesis of the topic label (e.g., Politics, Tech, Health).

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO)
1.  **Vertical Integration**: Look beyond the immediate story. If the story is about a local hospital (Local), does it represent a trend in National Healthcare Policy (Global)?
2.  **Category Refinement**: Choose the label that represents the highest "Level of Significance" in the article.

### PHASE 3: REFLECTIVE UPDATE (RUM)
1.  **Reflective Alignment**: Challenge your Phase 1 hypothesis and label the article with the correct topic.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field by explaining the logic of the chosen scale and domain.

## CONFIDENCE RUBRIC
- 90-100%: Topic is explicitly the central focus; total alignment with tool.
- 70-89%: Topic is dominant but intersects with sub-topics; tool results are supportive.
- 50-69%: Topic is present but ambiguous or shares equal weight with another domain.
- <50%: Content is generic or spans too many categories to classify reliably.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- The verdict must be a standardized topic label.
"""
)

In [8]:
intent_agent = Agent(
    name="Intent_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT specialist in identifying rhetorical intent and authorial goals.",
    instruction= """
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Transparency in identifying the author's underlying rhetorical goal (e.g., hidden persuasion).
- **MINIMIZE**: False categorization of "Opinion/Op-Ed" as "Deception" or "Satire" as "Informational."

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Goal Extraction**: Analyze the "Call to Action." What does the author want the reader to think, feel, or do after reading?
2.  **Linguistic Marker Identification**: Look for "Us vs. Them" framing, emotional appeals, or the presence/absence of verifiable citations.
3.  **Preliminary Intent**: Classify based on the four categories: [Inform, Persuade, Entertain, Deceive].

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO) - TOOL GROUNDING
1.  **Stakeholder Analysis**: Who benefits from this narrative? Does the text promote a specific commercial, political, or ideological interest?
2.  **Incentive Check**: If the article is factually weak but emotionally high-intensity, determine if the intent is to drive "clicks" (Entertain/Persuade) or to manipulate public perception (Deceive).

### PHASE 3: REFLECTIVE UPDATE (RUM) & DIALOGUE ALIGNMENT
1.  **Reflective Alignment**: Critically evaluate the article and determine if the linguistic markers suggest humor or irony that the tool ignored, and label the article with the correct intent.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field. Explain how the synthesis of analysis confirms the primary intent.

## CONFIDENCE RUBRIC
- 90-100%: Intent is explicit, consistent, and matches tool results.
- 70-89%: Intent is clear but contains minor stylistic nuance (e.g., informative text with slight persuasive leaning).
- 50-69%: Intent is ambiguous (e.g., "Advertorial" content mixing fact and persuasion).
- <50%: Intent is obscured by contradictory markers or heavy sarcasm.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary of the RUM process.
- Verdict must be strictly: [Inform, Persuade, Entertain, Deceive]."""
)

In [9]:
title_body_agent = Agent(
    name="Title_Body_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactorAnalysis,
    description="FCoT specialist in detecting semantic gaps between headlines and article content.",
    instruction="""
## LOCAL OBJECTIVE FUNCTION (LOF)
- **MAXIMIZE**: Detection of "headline-body gaps," bait-and-switch tactics, or semantic contradictions.
- **MINIMIZE**: False "Unrelated" labels for headlines that use metaphor or creative framing to describe the body content.

## FCoT REASONING PHASES

### PHASE 1: LOCAL THOUGHT UNIT (TU) - NARROW APERTURE
1.  **Direct Mapping**: Extract the core claim of the Headline. Scan the Body for direct supporting evidence of that specific claim.
2.  **Stance Alignment**: Does the body's tone match the headline's intensity? 
3.  **Preliminary Relationship**: Classify as [Agree, Discuss, Contradicts, Unrelated].

### PHASE 2: CONTEXT APERTURE EXPANSION (CAO)
1.  **Bait-and-Switch Audit**: Determine if the headline uses a "Curiosity Gap" that the body fails to resolve.
2.  **Logical Flow Test**: If you only read the body, would you have written that specific headline? If not, identify the point of divergence.

### PHASE 3: REFLECTIVE UPDATE (RUM)
1.  **Reflective Alignment**: Critically evaluate if the body is "Discussing" the topic but failing to "Agree" with a sensational headline. This is a common tactic for plausible deniability.
2.  **Final Synthesis**: Populate the `fcot_reasoning` field. Explain the logic of the relationship (e.g., "The headline negates the body by claiming X happened, while the body text only discusses the possibility of X").

## CONFIDENCE RUBRIC
- 90-100%: Relationship is explicit and supported/debunked by direct textual evidence.
- 70-89%: Relationship is clear; minor semantic nuances in metaphors or puns.
- 50-69%: The body explores the topic but the connection to the headline's specific claim is ambiguous.
- <50%: Headline and body appear disconnected or require heavy inference to link.

## OUTPUT RULES
- Populate the `fcot_reasoning` field with a concise 2-3 sentence summary.
- Verdict must be strictly: [Agree, Discuss, Contradicts, Unrelated].""",
)

In [10]:
from google.adk.agents import ParallelAgent

factor_squad = ParallelAgent(
    name="Factor_Squad",
    sub_agents=[
        sensationalism_agent, 
        stance_agent, 
        context_agent,
        news_coverage_agent, 
        intent_agent, 
        title_body_agent
    ]
)

In [11]:
synthesizer_agent = Agent(
    name="Final_Synthesizer",
    model=Gemini(model="gemini-3-flash-preview"),
    output_schema=FactCheckFinalReport,
    instruction="""
    You are the Final Synthesizer.
    
    1. **Recursive Synthesis**: Receive the 6 FactorAnalysis JSONs from the squad.
    2. **Inter-agent Reflectivity**: Identify if any agents disagree (e.g., if Context is Accurate but Intent is Deceive).
    3. **Retrospective Re-grounding**: If the Context_Veracity agent found a major factual error, force all other signals to be interpreted through that lens.
    4. **Output**: Generate a 'Human_Report' field using the following Markdown structure:

    1. **Executive Summary**: A bold verdict and 2-sentence 'Bottom Line Up Front'.
    2. **Factors Analysis Table**: A table with columns: | Factor | Verdict | Confidence | Key Evidence |.
    """
)

root_agent = SequentialAgent(
    name="Fractal_FactCheck_Framework",
    sub_agents=[factor_squad, synthesizer_agent]
)

In [12]:
article_body = "HONG KONG, Dec 2 (Reuters) - Hong Kong's leader said on Tuesday a judge-led committee will investigate the cause of the city's deadliest fire in decades and review government oversight of renovation practices linked to the blaze that killed at least 151 people.\nPolice have arrested 13 people for suspected manslaughter, and 12 others in a related corruption probe. Officials said substandard plastic mesh and insulation foam used during renovation works fueled the rapid spread of the fire across seven high-rise towers.\nAuthorities said they aim to avoid similar tragedies by examining how the fire spread so quickly and the oversight failures around building renovations.\n\nSEARCH AND INVESTIGATION\nInvestigators have combed most of the damaged towers, finding victims in stairwells and rooftops as they attempted to escape. Around 30 people remain missing.\nSome civic groups have demanded transparency and accountability, while police have warned against \"politicising\" the tragedy. A student was detained and later released, and media reports indicate others are under investigation for possible sedition.\nInternational rights groups argue the government's response reflects broader suppression of criticism.\n\nRESIDENTS WARNED PRIOR\nResidents of Wang Fuk Court had previously raised concerns about fire hazards and flammable materials used on scaffolding. Tests showed mesh samples did not meet fire-retardant standards.\nOfficials also reported foam insulation accelerated the fire and that alarms were malfunctioning.\nOver 1,500 residents have been displaced into temporary housing. Authorities are offering emergency funds and fast-tracked document replacement.\n\nVIGILS AND RECOVERY\nThousands across Hong Kong and cities like Tokyo, Taipei, and London have held vigils. Several victims were migrant domestic workers.\nThe search of the most heavily damaged towers may take weeks, as responders work through collapsed interiors."
article_title = "Hong Kong orders judge-led probe into fire that killed 151"
prompt = f"Title: {article_title}\nBody: {article_body}"

In [3]:
import json
import asyncio

def load_test_articles(path=gen_data_dir / "test_article_no_label.json"):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    return data.get("articles", [])

async def process_batch(articles_batch, batch_name="Batch"):
    print(f"=== Processing {batch_name} ({len(articles_batch)} articles) ===")
    
    for i, art in enumerate(articles_batch):
        headline = art.get('headline', 'No Title')
        print(f"\n[{batch_name}] Article {i+1}: {headline}")
        
        # Prepare Prompt
        prompt = (
            f"Headline: {headline}\n"
            f"Source: {art.get('news_source', 'Unknown')}\n"
            f"Author: {art.get('author', 'Unknown')}\n"
            f"Date: {art.get('date', 'Unknown')}\n\n"
            f"Body:\n{art.get('text', '')}"
        )
        
        # Run Agent using run_debug
        try:
            runner = InMemoryRunner(agent=root_agent)
            response = await runner.run_debug(prompt)
            print(response[-1].content.parts[0].text)
            
            # Access output based on ADK response structure
            # Assuming response has .output or similar
            if hasattr(response, 'output'):
                print(response.output)
            else:
                print(response)
                
        except Exception as e:
            print(f"Error running agent: {e}")

        print("-" * 50)
        
        # Sleep slightly to help with rate limits even within batch
        await asyncio.sleep(2)

# Load all data
all_articles = load_test_articles()
print(f"Total articles loaded: {len(all_articles)}")

Total articles loaded: 20


In [ ]:
await process_batch(all_articles[:2], "Batch 1")

=== Processing Batch 1 (2 articles) ===

[Batch 1] Article 1: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe

 ### Created new session: debug_session_id

User > Headline: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe
Source: Reuters
Author: Andrew Gray
Date: 12/02/2025

Body:
BRUSSELS, Dec 2 (Reuters) - However Donald Trump’s latest push to end the war in Ukraine pans out, Europe fears the prospect of a deal – sooner or later – that will not punish or weaken Russia as its leaders had hoped, placing the continent’s security in greater jeopardy.
Europe may well even have to accept a growing economic partnership between Washington, its traditional protector in the NATO alliance, and Moscow, which most European governments – and NATO itself – say is the greatest threat to European security.
Although Ukrainians and other Europeans managed to push back against parts of a 28-point U.S. plan to end the fighting that was seen as heavily pro-Russi

In [17]:
await process_batch(all_articles[2:4], "Batch 2")

=== Processing Batch 2 (2 articles) ===

[Batch 2] Article 1: Prison guard says Luigi Mangione, alleged CEO killer, spoke of 3D-printed gun

 ### Created new session: debug_session_id

User > Headline: Prison guard says Luigi Mangione, alleged CEO killer, spoke of 3D-printed gun
Source: Reuters
Author: Luc Cohen
Date: 12/01/2025

Body:
NEW YORK, Dec 1 (Reuters) - Luigi Mangione told a prison guard he had a 3D-printed gun in his backpack after his arrest for allegedly killing UnitedHealthcare CEO Brian Thompson, according to court testimony determining whether such statements and evidence can be used at trial.
Mangione, arrested in December 2024, faces murder and weapons charges. Prosecutors say a 3D-printed pistol, silencer, and incriminating journal writings were found in his backpack. His lawyers argue the search was unlawful and that he was questioned without proper Miranda warnings.

GUARD TESTIMONY
A prison guard testified Mangione volunteered information about the gun without bei

In [18]:
await process_batch(all_articles[4:6], "Batch 3")

=== Processing Batch 3 (2 articles) ===

[Batch 3] Article 1: HBO Max just messed up 'Mad Men' and it makes me sick

 ### Created new session: debug_session_id

User > Headline: HBO Max just messed up 'Mad Men' and it makes me sick
Source: Tom’s Guide
Author: Malcolm McMillan
Date: 12/03/2025

Body:
The 4K remaster of “Mad Men” on HBO Max includes visible production errors that fans quickly spotted — including a crew member operating a 'puke hose' in Season 1, Episode 7 during Roger Sterling’s vomiting scene.

THE 4K MISTAKE
In the original AMC broadcast, Sterling’s projectile vomiting was shown without revealing the practical effects equipment. The HBO Max version, however, accidentally includes a shot of a technician holding the hose.
Other mistakes include incorrect episode titles and stray environmental details like advertisements for SIM cards and Los Angeles restaurants appearing in scenes where they don’t belong.

WHAT CAUSED IT?
According to People Magazine, Lionsgate — the stu

In [22]:
await process_batch(all_articles[6:8], "Batch 4")

=== Processing Batch 4 (2 articles) ===

[Batch 4] Article 1: Biden secretly linked to Chinese financial influence operation

 ### Created new session: debug_session_id

User > Headline: Biden secretly linked to Chinese financial influence operation
Source: Conservative Brief
Author: Unknown
Date: Unknown

Body:
President Joe Biden has responded to Republicans’ claims that they have evidence that the Biden family received indirect payments from a Chinese company. Advertisement It happened on Friday night when an agitated President Biden responded to Russian President Vladimir Putin having an arrest warrant issued for him by the International Criminal Court as a reporter snuck in a question about his family’s business dealings.

Q Could you give us your reaction to the International Criminal Court issuing an arrest warrant for Vladimir Putin?
THE PRESIDENT: Well, I think it’s justified. But the question is, it’s not recognized internationally by us either. But I think it makes a very st

In [23]:
await process_batch(all_articles[8:10], "Batch 5")

=== Processing Batch 5 (2 articles) ===

[Batch 5] Article 1: How Modern 3D Laser Scanning Services Are Transforming Data Capture in Technology-Driven Industries

 ### Created new session: debug_session_id

User > Headline: How Modern 3D Laser Scanning Services Are Transforming Data Capture in Technology-Driven Industries
Source: News Examiner
Author: Jimmy Rustling
Date: 11/30/2025

Body:
In the fast-moving technology sector, 3D laser scanning services are redefining how companies collect, analyze, and apply digital information from physical objects. Using advanced laser scanning devices, teams can rapidly capture data from complex environments, producing a detailed point cloud that reflects real-world conditions with exceptional precision.

Precision Data for Engineering and Industrial Applications
Reverse Engineering and Advanced Digital Workflows
Reducing Project Costs and Improving Planning Accuracy

Compared to traditional measuring equipment, this cutting edge technology drastic

In [21]:
await process_batch(all_articles[10:15], "Batch 6")

=== Processing Batch 6 (5 articles) ===

[Batch 6] Article 1: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage

 ### Created new session: debug_session_id

User > Headline: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage
Source: Fox News
Author: Madison Colombo
Date: 11/11/2025

Body:
The International Olympic Committee is preparing to ban transgender women from female Olympic events following a scientific review concluding that biological males retain permanent physical advantages.
Former Olympic champion Caitlyn Jenner, a transgender woman, voiced support for restrictions, stating that biological males possess athletic advantages that hormone therapy cannot entirely eliminate.

JENNER'S POSITION
“I am a trans woman, but I am still biologically male,” Jenner said on 'America Reports.' She argued the policy is necessary to protect fairness in women’s sports.
Jenner noted tha

In [ ]:
await process_batch(all_articles[15:], "Batch 7")

=== Processing Batch 7 (5 articles) ===

[Batch 7] Article 1: You won’t believe what degrading practice the pope just condemned

 ### Created new session: debug_session_id

User > Headline: You won’t believe what degrading practice the pope just condemned
Source: The Guardian
Author: Unknown
Date: 10/09/2025

Body:
Pope Leo XIV condemned the practice of clickbait as “degrading” during a private audience with international newswire journalists. About 150 members of the global alliance Minds International attended the meeting.

POPE’S MESSAGE ON JOURNALISM
The pope urged journalists to resist sensationalism and prioritize communication as a public good. He argued that clickbait undermines transparency and objectivity.
He praised journalists reporting from conflict zones such as Gaza and Ukraine.

AI AND MEDIA
The pope warned about challenges posed by artificial intelligence, calling for oversight to prevent technological control by a small number of entities.
Key values he emphasized: tr